# For1Sight V2.7 – Machine Learning Development
## Stage 3: Train and Compare Three Race-Winner Models

### Prediction objective
Predict the Grand Prix winner after qualifying and before the race.

### Models
1. Logistic Regression
2. Random Forest
3. Gradient Boosting

### Evaluation design
- Train: 2024 (24 events)
- Validation / model selection: 2025 (24 events)
- Final test: 2026 (16 eligible events) — **not used in this notebook**

The 2026 season must remain unseen until Stage 4, after choosing the champion model using 2025 validation results.

### Important methodological choices
- One row represents one driver in one Grand Prix.
- The target is binary: `target = 1` for the race winner, otherwise `0`.
- Each Grand Prix has exactly one winner.
- Class imbalance is handled with balanced sample weights.
- Numeric imputation and scaling, categorical imputation and one-hot encoding are fitted inside each model pipeline using training data only.
- Driver-level scores are normalised within each Grand Prix to form a race-level winner-probability distribution.
- We report event-level Top-1, Top-3, winner log loss and multiclass Brier score, alongside row-level ROC-AUC and average precision.
- The starting-grid P1 baseline is included for comparison.

This notebook saves timestamped model artifacts locally. It does not write to HDFS, alter the V2.6 Parquet file, or modify Streamlit.

## 0. Environment and paths

Select the same **Python (For1Sight)** kernel used in Stages 1 and 2.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import warnings
import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path(r"C:\Users\ayepd\Desktop\For1Sight Project")
DATA_PATH = PROJECT_ROOT / "for1sight_warehouse" / "prediction_training_v2_6.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"V2.6 Parquet file not found: {DATA_PATH}")

print("Scikit-learn:", sklearn.__version__)
print("Dataset:", DATA_PATH)

Scikit-learn: 1.9.0
Dataset: C:\Users\ayepd\Desktop\For1Sight Project\for1sight_warehouse\prediction_training_v2_6.parquet


## 1. Import machine-learning tools

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib

print("Machine-learning libraries imported successfully.")

Machine-learning libraries imported successfully.


## 2. Load data and recreate the agreed eligible dataset

In [3]:
import re

def normalize_event_name(value):
    value = str(value or "").lower().strip()
    value = re.sub(r"\bgrand\s*prix\b", "", value)
    return re.sub(r"[^a-z0-9]", "", value)

def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})

df = pd.read_parquet(DATA_PATH, engine="pyarrow")

required = [
    "year", "event_name", "driver_code", "constructor",
    "finishing_position", "is_winner",
    "event_prediction_ready", "model_training_eligible",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

work = df.copy()
work["year"] = pd.to_numeric(work["year"], errors="coerce").astype("Int64")
work["event_key"] = work["event_name"].map(normalize_event_name)
work["target"] = pd.to_numeric(work["is_winner"], errors="coerce")
work["finishing_position_check"] = pd.to_numeric(work["finishing_position"], errors="coerce")
work["event_prediction_ready_check"] = to_bool(work["event_prediction_ready"])
work["model_training_eligible_check"] = to_bool(work["model_training_eligible"])

excluded_2026_keys = {"bahraininmalaysia", "unitedstates"}
excluded_mask = (work["year"] == 2026) & work["event_key"].isin(excluded_2026_keys)

eligible = work.loc[
    ~excluded_mask
    & work["event_prediction_ready_check"]
    & work["model_training_eligible_check"]
    & work["target"].isin([0, 1])
    & work["finishing_position_check"].notna()
].copy()

eligible["event_id"] = eligible["year"].astype(str) + "_" + eligible["event_key"].astype(str)

print("Eligible rows:", len(eligible))
print("Eligible events:", eligible["event_id"].nunique())
display(eligible.groupby("year").agg(
    rows=("event_id", "size"),
    events=("event_id", "nunique"),
    winner_labels=("target", "sum")
).reset_index().sort_values("year"))

Eligible rows: 1310
Eligible events: 64


,year,rows,events,winner_labels
0,2024,479,24,24
1,2025,479,24,24
2,2026,352,16,16


## 3. Validate one winner per event and create the chronological split

In [4]:
event_check = (
    eligible.groupby(["year", "event_id"])
    .agg(driver_rows=("driver_code", "size"),
         distinct_drivers=("driver_code", "nunique"),
         winner_labels=("target", "sum"))
    .reset_index()
)
event_check["valid"] = (
    (event_check["driver_rows"] == event_check["distinct_drivers"])
    & (event_check["winner_labels"] == 1)
)
display(event_check.loc[~event_check["valid"]])
if not event_check["valid"].all():
    raise ValueError("Event integrity failed. Stop before training.")

train_df = eligible.loc[eligible["year"] == 2024].copy()
valid_df = eligible.loc[eligible["year"] == 2025].copy()

if train_df.empty or valid_df.empty:
    raise ValueError("2024 training or 2025 validation data is empty.")

print("2024 training:", len(train_df), "driver rows /", train_df["event_id"].nunique(), "events")
print("2025 validation:", len(valid_df), "driver rows /", valid_df["event_id"].nunique(), "events")
print("2026 is deliberately not selected for model fitting or model selection.")

,year,event_id,driver_rows,distinct_drivers,winner_labels,valid


2024 training: 479 driver rows / 24 events
2025 validation: 479 driver rows / 24 events
2026 is deliberately not selected for model fitting or model selection.


## 4. Define the 30 candidate features

In [5]:
NUMERIC_FEATURES = [
    "round",
    "starting_grid",
    "best_practice_lap_s",
    "best_qualifying_lap_s",
    "best_quali_s1_s",
    "best_quali_s2_s",
    "best_quali_s3_s",
    "theoretical_best_sector_sum_s",
    "practice_to_qualifying_improvement_s",
    "practice_delta_to_event_best_s",
    "qualifying_delta_to_event_best_s",
    "event_best_practice_lap_s",
    "event_best_qualifying_lap_s",
    "qualifying_rank",
    "practice_timed_laps",
    "qualifying_timed_laps",
    "driver_prior_avg_finish_3",
    "driver_prior_avg_points_3",
    "driver_circuit_prior_avg_finish",
    "driver_circuit_prior_starts",
    "constructor_prior_avg_finish_3",
    "constructor_prior_avg_points_3",
    "constructor_circuit_prior_avg_finish",
    "constructor_circuit_prior_starts",
    "has_practice_data",
    "has_qualifying_data",
    "has_grid_data",
]
CATEGORICAL_FEATURES = [
    "driver_code_norm",
    "constructor",
    "event_norm",
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

missing_features = [c for c in FEATURE_COLUMNS if c not in eligible.columns]
if missing_features:
    raise ValueError(f"Candidate feature columns missing: {missing_features}")

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total:", len(FEATURE_COLUMNS))

Numeric features: 27
Categorical features: 3
Total: 30


## 5. Build a leakage-safe preprocessing pipeline

All preprocessing is fitted only on the 2024 training data:
- Numeric values: median imputation and standardisation.
- Numeric missingness: explicit missing flags for every numeric feature.
- Categorical values: most-frequent imputation and one-hot encoding.
- Unknown drivers, constructors or circuits: ignored safely by the encoder.

A dense one-hot output is used because Scikit-learn's `GradientBoostingClassifier` expects dense input.

In [6]:
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # Compatibility with older scikit-learn releases
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", encoder),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("numeric_missing_flags", MissingIndicator(features="all"), NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## 6. Prepare training and validation inputs

In [7]:
X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df["target"].astype(int).copy()

X_valid = valid_df[FEATURE_COLUMNS].copy()
y_valid = valid_df["target"].astype(int).copy()

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("Training winner labels:", int(y_train.sum()))
print("Training non-winner labels:", int((y_train == 0).sum()))

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
print("Balanced sample weights calculated.")

Training shape: (479, 30)
Validation shape: (479, 30)
Training winner labels: 24
Training non-winner labels: 455
Balanced sample weights calculated.


## 7. Define the three candidate models

These are deliberately regularised starting configurations, not an exhaustive hyperparameter search. We have only 24 training races, so overly complex tuning could overfit the small training sample.

In [8]:
models = {
    "Logistic Regression": LogisticRegression(
        C=0.3,
        max_iter=5000,
        solver="lbfgs",
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=5,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=120,
        learning_rate=0.03,
        max_depth=2,
        min_samples_leaf=5,
        subsample=0.8,
        random_state=42,
    ),
}

print("Models ready:", list(models))

Models ready: ['Logistic Regression', 'Random Forest', 'Gradient Boosting']


## 8. Train models using 2024 only

Each model receives the same balanced sample weights. The 2025 validation data is not used to fit preprocessing or model parameters.

In [9]:
trained_models = {}
training_rows = []

for model_name, estimator in models.items():
    print(f"Training {model_name} ...")
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator),
    ])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        pipeline.fit(X_train, y_train, classifier__sample_weight=sample_weights)
    trained_models[model_name] = pipeline
    training_rows.append({
        "model": model_name,
        "training_rows": len(X_train),
        "training_events": train_df["event_id"].nunique(),
        "training_winner_rows": int(y_train.sum()),
        "warnings": " | ".join(str(w.message) for w in caught) if caught else "None",
    })
    print(f"Completed: {model_name}")

display(pd.DataFrame(training_rows))

Training Logistic Regression ...
Completed: Logistic Regression
Training Random Forest ...
Completed: Random Forest
Training Gradient Boosting ...
Completed: Gradient Boosting


,model,training_rows,training_events,training_winner_rows,warnings
0,Logistic Regression,479,24,24,Skipping features without any observed values:...
1,Random Forest,479,24,24,Skipping features without any observed values:...
2,Gradient Boosting,479,24,24,Skipping features without any observed values:...


## 9. Convert driver scores into race-level winner probabilities

A binary classifier estimates a win score for each driver row. Because each Grand Prix has exactly one winner, we convert the scores into a probability distribution within each event.

We:
1. Obtain each driver's predicted positive-class probability.
2. Convert it to a log-odds score.
3. Apply a softmax within each Grand Prix.
4. Ensure the driver probabilities for each event sum to 1.

These are model-derived relative race probabilities. They should not be described as perfectly calibrated real-world probabilities without further calibration analysis.

In [10]:
def event_probability_predictions(model, frame):
    X = frame[FEATURE_COLUMNS].copy()
    raw_p = model.predict_proba(X)[:, 1]
    result = frame[["event_id", "year", "event_name", "driver_code", "target"]].copy()
    result["raw_win_score"] = raw_p

    eps = 1e-8
    clipped = np.clip(raw_p, eps, 1 - eps)
    result["log_odds_score"] = np.log(clipped / (1 - clipped))

    def softmax_group(values):
        values = np.asarray(values, dtype=float)
        shifted = values - np.max(values)
        exp_values = np.exp(shifted)
        return exp_values / exp_values.sum()

    result["race_win_probability"] = (
        result.groupby("event_id")["log_odds_score"]
        .transform(softmax_group)
    )
    return result

validation_predictions = {
    name: event_probability_predictions(model, valid_df)
    for name, model in trained_models.items()
}

for name, pred in validation_predictions.items():
    sums = pred.groupby("event_id")["race_win_probability"].sum()
    print(name, "| probability sum min/max:",
          round(sums.min(), 8), round(sums.max(), 8))

Logistic Regression | probability sum min/max: 1.0 1.0
Random Forest | probability sum min/max: 1.0 1.0
Gradient Boosting | probability sum min/max: 1.0 1.0


C:\Users\ayepd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['driver_circuit_prior_avg_finish' 'constructor_circuit_prior_avg_finish']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\ayepd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['driver_circuit_prior_avg_finish' 'constructor_circuit_prior_avg_finish']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\ayepd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features

## 10. Evaluate at Grand Prix level and driver-row level

In [11]:
def evaluate_event_predictions(pred):
    event_rows = []
    for event_id, group in pred.groupby("event_id"):
        group = group.sort_values("race_win_probability", ascending=False).reset_index(drop=True)
        winner_positions = np.flatnonzero(group["target"].to_numpy() == 1)
        if len(winner_positions) != 1:
            raise ValueError(f"{event_id} does not have exactly one winner.")
        winner_index = int(winner_positions[0])
        winner_probability = float(group.loc[winner_index, "race_win_probability"])
        event_rows.append({
            "event_id": event_id,
            "actual_winner": group.loc[winner_index, "driver_code"],
            "predicted_winner": group.loc[0, "driver_code"],
            "top1_correct": winner_index == 0,
            "top3_correct": winner_index < 3,
            "winner_rank": winner_index + 1,
            "winner_probability": winner_probability,
            "winner_log_loss": -np.log(max(winner_probability, 1e-15)),
            "multiclass_brier": float(
                np.sum(
                    (group["race_win_probability"].to_numpy()
                     - group["target"].to_numpy()) ** 2
                )
            ),
            "reciprocal_rank": 1.0 / (winner_index + 1),
        })
    event_metrics = pd.DataFrame(event_rows)
    row_auc = roc_auc_score(pred["target"], pred["raw_win_score"])
    row_ap = average_precision_score(pred["target"], pred["raw_win_score"])
    summary = {
        "Top-1 accuracy (%)": event_metrics["top1_correct"].mean() * 100,
        "Top-3 accuracy (%)": event_metrics["top3_correct"].mean() * 100,
        "Winner log loss (lower better)": event_metrics["winner_log_loss"].mean(),
        "Multiclass Brier (lower better)": event_metrics["multiclass_brier"].mean(),
        "Mean reciprocal rank": event_metrics["reciprocal_rank"].mean(),
        "Driver-row ROC-AUC": row_auc,
        "Driver-row average precision": row_ap,
        "Events evaluated": len(event_metrics),
    }
    return summary, event_metrics

validation_metrics = {}
validation_event_details = {}
for name, pred in validation_predictions.items():
    summary, details = evaluate_event_predictions(pred)
    validation_metrics[name] = summary
    validation_event_details[name] = details

metrics_df = pd.DataFrame(validation_metrics).T.reset_index(names="Model")
display(metrics_df.round(4))

,Model,Top-1 accuracy (%),Top-3 accuracy (%),Winner log loss (lower better),Multiclass Brier (lower better),Mean reciprocal rank,Driver-row ROC-AUC,Driver-row average precision,Events evaluated
0,Logistic Regression,45.8333,91.6667,1.0977,0.6076,0.7014,0.9638,0.5590,24.0
1,Random Forest,50.0000,95.8333,1.2163,0.6216,0.7118,0.9668,0.5811,24.0
2,Gradient Boosting,58.3333,95.8333,1.0769,0.5611,0.7535,0.9625,0.5447,24.0


## 11. Calculate the 2025 starting-grid baseline

This baseline predicts the driver starting from P1 as the winner. Top-3 indicates whether the actual winner started in the first three grid positions.

In [12]:
def starting_grid_baseline(frame):
    event_results = []
    for event_id, group in frame.groupby("event_id"):
        valid_grid = group.dropna(subset=["starting_grid"]).copy()
        if valid_grid.empty:
            continue
        winner = valid_grid.loc[valid_grid["target"] == 1]
        if len(winner) != 1:
            continue
        winner_grid = float(winner.iloc[0]["starting_grid"])
        predicted_grid = float(valid_grid["starting_grid"].min())
        event_results.append({
            "event_id": event_id,
            "actual_winner_grid": winner_grid,
            "top1_correct": winner_grid == predicted_grid,
            "top3_correct": winner_grid <= 3,
        })
    return pd.DataFrame(event_results)

baseline_details = starting_grid_baseline(valid_df)
baseline_metrics = {
    "Model": "Starting-grid P1 baseline",
    "Top-1 accuracy (%)": baseline_details["top1_correct"].mean() * 100,
    "Top-3 accuracy (%)": baseline_details["top3_correct"].mean() * 100,
    "Winner log loss (lower better)": np.nan,
    "Multiclass Brier (lower better)": np.nan,
    "Mean reciprocal rank": np.nan,
    "Driver-row ROC-AUC": np.nan,
    "Driver-row average precision": np.nan,
    "Events evaluated": len(baseline_details),
}

comparison_df = pd.concat(
    [metrics_df, pd.DataFrame([baseline_metrics])],
    ignore_index=True
)
display(comparison_df.round(4))

,Model,Top-1 accuracy (%),Top-3 accuracy (%),Winner log loss (lower better),Multiclass Brier (lower better),Mean reciprocal rank,Driver-row ROC-AUC,Driver-row average precision,Events evaluated
0,Logistic Regression,45.8333,91.6667,1.0977,0.6076,0.7014,0.9638,0.5590,24.0
1,Random Forest,50.0000,95.8333,1.2163,0.6216,0.7118,0.9668,0.5811,24.0
2,Gradient Boosting,58.3333,95.8333,1.0769,0.5611,0.7535,0.9625,0.5447,24.0
3,Starting-grid P1 baseline,66.6667,95.8333,NaN,NaN,NaN,NaN,NaN,24.0


## 12. Select a provisional champion using 2025 validation only

Selection priority:
1. Lowest event-level winner log loss.
2. Higher event-level Top-1 accuracy.
3. Higher Top-3 accuracy.

The starting-grid baseline is a benchmark, not a trainable model. The 2026 test season is not used to select the champion.

In [13]:
model_only = metrics_df.copy()
model_only = model_only.sort_values(
    by=[
        "Winner log loss (lower better)",
        "Top-1 accuracy (%)",
        "Top-3 accuracy (%)",
    ],
    ascending=[True, False, False],
).reset_index(drop=True)

display(model_only)

CHAMPION_NAME = str(model_only.loc[0, "Model"])
CHAMPION_MODEL = trained_models[CHAMPION_NAME]

print("Provisional champion based on 2025 validation:", CHAMPION_NAME)
print("This is provisional until we inspect the metrics and model behaviour.")

,Model,Top-1 accuracy (%),Top-3 accuracy (%),Winner log loss (lower better),Multiclass Brier (lower better),Mean reciprocal rank,Driver-row ROC-AUC,Driver-row average precision,Events evaluated
0,Gradient Boosting,58.333333,95.833333,1.076886,0.561133,0.753472,0.962454,0.544676,24.0
1,Logistic Regression,45.833333,91.666667,1.097704,0.607574,0.701389,0.963828,0.558965,24.0
2,Random Forest,50.000000,95.833333,1.216278,0.621596,0.711806,0.966850,0.581108,24.0


Provisional champion based on 2025 validation: Gradient Boosting
This is provisional until we inspect the metrics and model behaviour.


## 13. Inspect the champion's event-by-event predictions

In [14]:
champion_event_results = validation_event_details[CHAMPION_NAME].copy()
champion_event_results = champion_event_results.sort_values(
    ["top1_correct", "winner_probability"],
    ascending=[True, True]
)
display(champion_event_results)

,event_id,actual_winner,predicted_winner,top1_correct,top3_correct,winner_rank,winner_probability,winner_log_loss,multiclass_brier,reciprocal_rank
16,2025_miami,PIA,VER,False,False,4,0.034926,3.354527,1.293448,0.250000
18,2025_qatar,VER,PIA,False,True,3,0.088551,2.424174,1.175984,0.333333
19,2025_saudiarabian,PIA,VER,False,True,3,0.112819,2.181973,1.341321,0.333333
6,2025_british,NOR,VER,False,True,3,0.148138,1.909612,1.028825,0.333333
21,2025_sopaulo,NOR,LEC,False,True,2,0.190348,1.658900,1.100935,0.500000
10,2025_emiliaromagna,VER,RUS,False,True,3,0.219552,1.516168,0.805570,0.333333
22,2025_spanish,PIA,VER,False,True,2,0.250769,1.383224,0.849383,0.500000
5,2025_belgian,PIA,NOR,False,True,2,0.284041,1.258635,0.900688,0.500000
14,2025_lasvegas,VER,NOR,False,True,2,0.331848,1.103078,0.585895,0.500000
9,2025_dutch,PIA,NOR,False,True,2,0.415872,0.877377,0.527202,0.500000


## 14. Inspect feature importance / model coefficients

Tree models provide impurity-based feature importance. Logistic Regression provides signed coefficients. These are useful diagnostics, not proof of causation.

In [15]:
def get_model_feature_importance(pipeline, model_name):
    prep = pipeline.named_steps["preprocessor"]
    clf = pipeline.named_steps["classifier"]
    feature_names = prep.get_feature_names_out()

    if hasattr(clf, "feature_importances_"):
        values = clf.feature_importances_
        direction = ["importance"] * len(values)
    elif hasattr(clf, "coef_"):
        coef = np.asarray(clf.coef_).ravel()
        values = np.abs(coef)
        direction = ["absolute_coefficient"] * len(values)
    else:
        return pd.DataFrame()

    output = pd.DataFrame({
        "model": model_name,
        "transformed_feature": feature_names,
        "importance": values,
        "measure": direction,
    })
    return output.sort_values("importance", ascending=False)

importance_tables = []
for name, pipeline in trained_models.items():
    table = get_model_feature_importance(pipeline, name)
    importance_tables.append(table.head(25))

importance_df = pd.concat(importance_tables, ignore_index=True)
display(importance_df.groupby("model", group_keys=False).head(10))

,model,transformed_feature,importance,measure
0,Logistic Regression,numeric__qualifying_rank,1.438319,absolute_coefficient
1,Logistic Regression,categorical__driver_code_norm_VER,1.003155,absolute_coefficient
2,Logistic Regression,numeric__starting_grid,0.916884,absolute_coefficient
3,Logistic Regression,numeric__driver_prior_avg_finish_3,0.877297,absolute_coefficient
4,Logistic Regression,categorical__driver_code_norm_PER,0.664589,absolute_coefficient
5,Logistic Regression,categorical__event_norm_miami,0.526054,absolute_coefficient
6,Logistic Regression,numeric__driver_prior_avg_points_3,0.438307,absolute_coefficient
7,Logistic Regression,numeric__qualifying_delta_to_event_best_s,0.375447,absolute_coefficient
8,Logistic Regression,categorical__constructor_Red Bull,0.338566,absolute_coefficient
9,Logistic Regression,numeric__practice_to_qualifying_improvement_s,0.331957,absolute_coefficient


## 15. Save timestamped model artifacts and validation outputs

Files are saved to a new timestamped folder under `models/race_winner_v2_7`. Existing model files are not overwritten. No 2026 predictions are generated or saved in this stage.

In [16]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_ROOT / "models" / "race_winner_v2_7" / f"stage3_validation_{run_id}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

for name, model in trained_models.items():
    safe_name = name.lower().replace(" ", "_")
    joblib.dump(model, OUTPUT_DIR / f"{safe_name}.joblib")

joblib.dump(CHAMPION_MODEL, OUTPUT_DIR / "provisional_champion_2025_validation.joblib")

comparison_df.to_csv(OUTPUT_DIR / "model_comparison_validation_2025.csv", index=False)
champion_event_results.to_csv(OUTPUT_DIR / "champion_event_predictions_2025.csv", index=False)
importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

all_validation_predictions = []
for name, pred in validation_predictions.items():
    temp = pred.copy()
    temp["model"] = name
    all_validation_predictions.append(temp)
pd.concat(all_validation_predictions, ignore_index=True).to_csv(
    OUTPUT_DIR / "driver_probabilities_validation_2025.csv", index=False
)

manifest = {
    "project": "For1Sight",
    "version": "V2.7",
    "stage": "Stage 3 - model training and 2025 validation",
    "run_id": run_id,
    "training_season": 2024,
    "validation_season": 2025,
    "final_test_season": 2026,
    "final_test_used": False,
    "excluded_2026_event_keys": sorted(excluded_2026_keys),
    "models": list(trained_models.keys()),
    "provisional_champion": CHAMPION_NAME,
    "feature_columns": FEATURE_COLUMNS,
    "random_state": 42,
    "scikit_learn_version": sklearn.__version__,
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print("Saved outputs to:", OUTPUT_DIR)
print("2026 final test was not used.")

Saved outputs to: C:\Users\ayepd\Desktop\For1Sight Project\models\race_winner_v2_7\stage3_validation_20261113_220702
2026 final test was not used.


## 16. Stage 3 completion checklist

Before moving to Stage 4, review:
- [ ] All three models trained without errors.
- [ ] The 2025 validation metrics table is available.
- [ ] Compare each model against the P1 starting-grid baseline.
- [ ] Review event-by-event winner ranks and probabilities.
- [ ] Review feature importance / coefficients.
- [ ] Check warnings and any unexpected model behaviour.
- [ ] Confirm the manifest states `final_test_used: false`.

### Important interpretation
There are only 24 training events and 24 validation events. Results can vary substantially from race to race. Do not claim that a model is definitively superior from this small sample alone. We will preserve 2026 as the final unseen test and report the limitation transparently.

**Next stage:** Stage 4 – inspect and select the model, then perform the one-time final evaluation on the 2026 season.